# Аудит статей базы знаний

Ноутбук проверяет формат и обязательные метаданные, сводит корпус по языкам, владельцам и лицензиям, отмечает дубликаты и статьи, которым пора на ручной пересмотр. Самих статей в репозитории пока нет. Добавляйте только одобренные данные с подтверждённым правом использования.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CORPUS_PATH = ROOT / 'data' / 'knowledge' / 'articles.jsonl'
REPORT_PATH = ROOT / 'reports' / 'knowledge_audit.json'
REVIEW_AFTER_DAYS = 365


In [ ]:
from src.rag.audit import audit_corpus
import json

audit_report = audit_corpus(CORPUS_PATH, review_after_days=REVIEW_AFTER_DAYS)
print(json.dumps(audit_report, ensure_ascii=False, indent=2))


## Сводка и замечания для проверки

Если формат некорректен, исправьте строки по тексту ошибки и повторите ячейку. Дубликаты — повод проверить актуальность и ID, но не автоматически удалить статью: похожий заголовок может относиться к разным продуктам. Статьи в `review_due` нужно проверить у указанного владельца.


In [ ]:
if audit_report['valid']:
    import pandas as pd
    summary = pd.DataFrame([audit_report['summary']])
    review_queue = pd.DataFrame(audit_report['review_due'])
    duplicate_titles = pd.DataFrame(audit_report['duplicate_titles'])
    duplicate_contents = pd.DataFrame(audit_report['duplicate_contents'])
    display(summary)
    display(review_queue)
    display(duplicate_titles)
    display(duplicate_contents)
else:
    print('Аудит не пройден: проверьте формат и обязательные поля.')


In [ ]:
if audit_report['valid']:
    REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    REPORT_PATH.write_text(
        json.dumps(audit_report, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    print(f'Отчёт записан: {REPORT_PATH}')
else:
    print('Отчёт не сохранён: входной корпус не прошёл проверку.')


Порог 365 дней — напоминание о проверке, а не автоматическая оценка актуальности бизнес-правила. Владелец должен подтвердить статью или внести новую дату. Итоговый JSON-отчёт можно просмотреть и передать ответственному за базу знаний.
